In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("./augmented_reranker_dataset")

In [ ]:
df = dataset.to_pandas()

In [ ]:
df.head()

In [ ]:
print(df.iloc[0]["text"])


In [ ]:
gid = "760d366ab61ef27e2bc7135ef13f9e57"

for i, q in enumerate(df.loc[df["query_group_id"] == gid, "query_text"]):
    print(f"{i}: {q}")


In [ ]:
gid = "760d366ab61ef27e2bc7135ef13f9e57"

rows = df.loc[df["query_group_id"] == gid, "candidate_scores_k"]

for i, scores in enumerate(rows):
    uniq = sorted(set(scores))
    print(f"Variant {i}: unique scores = {uniq}")


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-27b-it",
    max_seq_length = 16384,
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit = True,
    full_finetuning = False, # [NEW!] We have full finetuning now!
)
model.eval()


In [ ]:
@torch.no_grad()
def generate_query(prompt, max_new_tokens=64):
    inputs = tokenizer(
        text=prompt,               # 👈 IMPORTANT
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.strip()


In [ ]:
def rewrite_query_group(df_group, abstract_lookup):
    """
    df_group: pandas DataFrame for ONE query_group_id
    returns: list of dicts with rewritten queries
    """

    original_query = df_group.iloc[0]["query_text"]
    results = []

    for variant_idx, row in df_group.iterrows():
        scores = row["candidate_scores_k"]
        pool   = row["candidate_pool_k"]

        # highest scoring candidate
        top_i = max(range(len(scores)), key=lambda i: scores[i])
        top_doc_id = pool[top_i]
        top_doc = abstract_lookup[int(top_doc_id)]

        # truncate doc aggressively (important)
        top_doc = " ".join(top_doc.split()[:150])

        if len(results) == 0:
            # Variant A: unchanged
            results.append({
                "query_type": "original",
                "query_text": original_query,
            })
        elif len(results) == 1:
            # Variant B: focused
            prompt = f"""
You are rewriting a search query to be MORE SPECIFIC.

Original query:
"{original_query}"

Relevant document:
"{top_doc}"

Rewrite the query to be more specific and precise, focusing on the key concepts in the document.
Limit the result to 1–2 sentences.
Return ONLY the rewritten query.
""".strip()

            rewritten = generate_query(prompt)
            results.append({
                "query_type": "focused",
                "query_text": rewritten,
            })
        else:
            # Variant C: broader
            prompt = f"""
You are rewriting a search query to be MORE GENERAL.

Original query:
"{original_query}"

Relevant document:
"{top_doc}"

Rewrite the query to be broader and more general, covering a wider interpretation of the topic.
Limit the result to 1–2 sentences.
Return ONLY the rewritten query.
""".strip()

            rewritten = generate_query(prompt)
            results.append({
                "query_type": "broad",
                "query_text": rewritten,
            })

    return results


In [ ]:
import pandas as pd

corpus_df = pd.read_json("splits/Corpus.json")

In [ ]:
print("Creating abstract_id -> text lookup dictionary...")
abstract_lookup = {}
for _, row in corpus_df.iterrows():
    title = row['title'].replace("_", " ")
    text = f"Title: {title}\nAbstract: {row['original_abstract']}"
    abstract_lookup[int(row['abstract_id'])] = text

print(f"Created lookup for {len(abstract_lookup)} abstracts.")

In [ ]:
rewritten_queries = []

for gid, group in df.groupby("query_group_id"):
    rewritten = rewrite_query_group(group, abstract_lookup)
    for r in rewritten:
        r["query_group_id"] = gid
        rewritten_queries.append(r)


In [ ]:
rewritten_df = pd.DataFrame(rewritten_queries)


In [ ]:
rewritten_df.head()

In [ ]:
rewritten_df.iloc[1]['query_text']

In [ ]:
rewritten_df["query"] = rewritten_df["query_text"].str.split(
    pat="Return ONLY the rewritten query.\n\n", n=1
).str[-1].str.strip()


In [ ]:
rewritten_df.iloc[1]["query"]


In [ ]:
rewritten_df["query"] = rewritten_df["query"].str.strip('"').str.strip("'")


In [ ]:
rewritten_df.head()

In [ ]:
rewritten_df.to_parquet("rewritten_queries.parquet", index=False)


In [ ]:
rewritten_df.to_json("rewritten_queries.jsonl", orient="records", lines=True)

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(rewritten_df)


In [ ]:
dataset.save_to_disk("./rewritten_queries_dataset")
